# Data Cleaning & Reporting Automation
**Author:** Abhinav Gaur
**Dataset:** Superstore Sales (Kaggle: rohitsahoo/sales-forecasting)
**Goal:** Automate the cleaning of messy sales data (missing values, duplicates, inconsistent entries) and automatically generate an Excel report and a visual summary.

In [1]:
import pandas as pd
import numpy as np

raw = pd.read_csv("train.csv")
print(raw.shape)
raw.head()

(9800, 18)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


## Creating a messy dataset
The original dataset is already fairly clean, so to demonstrate the cleaning pipeline a messy copy was created **on purpose** with these problems:
- About 5% missing values in Sales, Region, Segment, Category and Customer Name
- Duplicate rows (300 rows were copied into the data)
- Inconsistent text: Region in capitals, extra spaces in Category, Segment in lowercase
- Mixed date formats (DD/MM/YYYY and YYYY/MM/DD)

In [2]:
np.random.seed(42)
messy = raw.copy()

# 1. Missing values (about 5%)
for col in ["Sales", "Region", "Segment", "Category", "Customer Name"]:
    idx = messy.sample(frac=0.05).index
    messy.loc[idx, col] = np.nan

# 2. Duplicate rows
messy = pd.concat([messy, messy.sample(300)], ignore_index=True)

# 3. Inconsistent text (case and extra spaces)
idx = messy.sample(frac=0.10).index
messy.loc[idx, "Region"] = messy.loc[idx, "Region"].str.upper()
idx = messy.sample(frac=0.10).index
messy.loc[idx, "Category"] = " " + messy.loc[idx, "Category"].astype(str) + " "
idx = messy.sample(frac=0.05).index
messy.loc[idx, "Segment"] = messy.loc[idx, "Segment"].str.lower()

# 4. Mixed date formats
idx = messy.sample(frac=0.10).index
messy.loc[idx, "Order Date"] = pd.to_datetime(
    messy.loc[idx, "Order Date"], dayfirst=True).dt.strftime("%Y/%m/%d")

messy.to_csv("messy_sales.csv", index=False)
print(messy.shape)

(10100, 18)


In [3]:
def quality_report(d):
    return pd.DataFrame({
        "Missing": d.isnull().sum(),
        "Missing %": (d.isnull().mean() * 100).round(1)
    }).query("Missing > 0")

print("Duplicate rows:", messy.duplicated().sum())
print("Unique Region values:", messy["Region"].unique())
print("Unique Category values:", messy["Category"].unique())
quality_report(messy)

Duplicate rows: 144
Unique Region values: ['South' 'West' nan 'CENTRAL' 'Central' 'WEST' 'East' 'EAST' 'SOUTH']
Unique Category values: ['Furniture' 'Office Supplies' 'Technology' ' Technology ' nan
 ' Furniture ' ' Office Supplies ' ' nan ']


,Missing,Missing %
Customer Name,504,5.0
Segment,506,5.0
Postal Code,11,0.1
Region,501,5.0
Category,455,4.5
Sales,497,4.9


## Data quality check (before cleaning)
The check found the following problems:
- **Duplicate rows:** 144 exact duplicates
- **Missing values:**

| Column | Missing | Missing % |
|---|---|---|
| Customer Name | 504 | 5.0% |
| Segment | 506 | 5.0% |
| Region | 501 | 5.0% |
| Category | 455 | 4.5% |
| Sales | 497 | 4.9% |
| Postal Code | 11 | 0.1% |

- **Inconsistent text:** Region appears in different forms, such as "CENTRAL" and "Central", and "WEST" and "West". Category has extra spaces around values (for example " Furniture ").
- **Dates:** two different date formats are mixed (DD/MM/YYYY and YYYY/MM/DD).

The 11 missing Postal Codes were already in the original dataset. The other gaps were added on purpose.

## Automated cleaning function
The function `clean_data()` fixes each problem in a fixed order:
1. **Duplicates:** exact duplicate rows are removed.
2. **Inconsistent text:** extra spaces are trimmed and text is converted to Title Case, so "WEST", "west" and " West " all become "West".
3. **Dates:** both date formats are parsed into one proper date type. Rows with an unreadable date are dropped.
4. **Missing numeric values (Sales):** filled with the median of the same Sub-Category, which is more realistic than one overall value.
5. **Missing categories (Region, Segment, Category):** filled with the most common value (mode).
6. **Missing names and postal codes:** filled with "Unknown" and 0.

The function also returns a log of how many rows were affected.

In [10]:
def clean_data(df):
    log = {}
    df = df.copy()
    log["rows_before"] = len(df)

    # 1. Remove exact duplicates
    log["duplicates_removed"] = int(df.duplicated().sum())
    df = df.drop_duplicates()

    # 2. Clean text columns: trim spaces, turn "nan"/"none"/empty text into real missing values
    for col in ["Region", "Category", "Segment", "Customer Name"]:
        df[col] = df[col].astype("string").str.strip()
        fake_missing = df[col].str.lower().isin(["nan", "none", "null", ""])
        df[col] = df[col].mask(fake_missing)
    for col in ["Region", "Category", "Segment"]:
        df[col] = df[col].str.title()

    # 3. Fix mixed date formats
    d1 = pd.to_datetime(df["Order Date"], format="%d/%m/%Y", errors="coerce")
    d2 = pd.to_datetime(df["Order Date"], format="%Y/%m/%d", errors="coerce")
    df["Order Date"] = d1.fillna(d2)
    log["bad_dates_dropped"] = int(df["Order Date"].isnull().sum())
    df = df.dropna(subset=["Order Date"])

    # 4. Remove duplicates again, now that text and dates are standardized
    log["duplicates_removed_after_standardizing"] = int(df.duplicated().sum())
    df = df.drop_duplicates()

    # 5. Fill missing values
    log["missing_sales_filled"] = int(df["Sales"].isnull().sum())
    df["Sales"] = df["Sales"].fillna(
        df.groupby("Sub-Category")["Sales"].transform("median"))
    df["Sales"] = df["Sales"].fillna(df["Sales"].median())

    for col in ["Region", "Segment", "Category"]:
        log[f"missing_{col.lower()}_filled"] = int(df[col].isnull().sum())
        df[col] = df[col].fillna(df[col].mode()[0])
    df["Customer Name"] = df["Customer Name"].fillna("Unknown")
    df["Postal Code"] = df["Postal Code"].fillna(0)

    # 6. Add helper columns
    df["Year"] = df["Order Date"].dt.year
    df["Month"] = df["Order Date"].dt.to_period("M").astype(str)

    log["rows_after"] = len(df)
    return df.reset_index(drop=True), log

clean, log = clean_data(messy)
log

{'rows_before': 10100,
 'duplicates_removed': 144,
 'bad_dates_dropped': 0,
 'duplicates_removed_after_standardizing': 156,
 'missing_sales_filled': 490,
 'missing_region_filled': 490,
 'missing_segment_filled': 490,
 'missing_category_filled': 490,
 'rows_after': 9800}

In [11]:
print("Duplicates:", clean.duplicated().sum())
print("Regions:", clean["Region"].unique())
print("Categories:", clean["Category"].unique())
print("Date type:", clean["Order Date"].dtype)
quality_report(clean)

Duplicates: 0
Regions: <StringArray>
['South', 'West', 'Central', 'East']
Length: 4, dtype: string
Categories: <StringArray>
['Furniture', 'Office Supplies', 'Technology']
Length: 3, dtype: string
Date type: datetime64[ns]


,Missing,Missing %


## Data quality check (after cleaning)
After cleaning:
- **Duplicates:** 0
- **Missing values:** none
- **Region** has 4 clean values, **Category** has 3, and **Segment** has 3.
- **Order Date** is a proper date column.
- The dataset went from [rows before] rows to [rows after] rows, with [number] duplicates removed.

## Automated Excel report
The function `generate_excel_report()` creates `Sales_Report.xlsx` with five sheets:
- **Summary:** total sales, total orders, average order value, unique customers
- **By Region** and **By Category:** sales totals
- **Monthly:** sales per month
- **Top 10 Products:** the best-selling products

In [6]:
def generate_excel_report(df, path="Sales_Report.xlsx"):
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        summary = pd.DataFrame({
            "Metric": ["Total Sales", "Total Orders", "Avg Order Value", "Unique Customers"],
            "Value": [round(df["Sales"].sum(), 2), df["Order ID"].nunique(),
                      round(df["Sales"].sum() / df["Order ID"].nunique(), 2),
                      df["Customer ID"].nunique()]
        })
        summary.to_excel(writer, sheet_name="Summary", index=False)
        df.groupby("Region")["Sales"].sum().round(2).reset_index().to_excel(writer, "By Region", index=False)
        df.groupby("Category")["Sales"].sum().round(2).reset_index().to_excel(writer, "By Category", index=False)
        df.groupby("Month")["Sales"].sum().round(2).reset_index().to_excel(writer, "Monthly", index=False)
        df.groupby("Product Name")["Sales"].sum().nlargest(10).round(2).reset_index().to_excel(writer, "Top 10 Products", index=False)
    return path

generate_excel_report(clean)

/tmp/ipykernel_1004/3035551854.py:10: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.groupby("Region")["Sales"].sum().round(2).reset_index().to_excel(writer, "By Region", index=False)
/tmp/ipykernel_1004/3035551854.py:11: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.groupby("Category")["Sales"].sum().round(2).reset_index().to_excel(writer, "By Category", index=False)
/tmp/ipykernel_1004/3035551854.py:12: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.groupby("Month")["Sales"].sum().round(2).reset_index().to_excel(writer, "Monthly", index=False)
/tmp/ipykernel_1004/3035551854.py:13: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-o

'Sales_Report.xlsx'

## Automated visual summary
The function `generate_visual_report()` creates `Sales_Visual_Report.html`, a dashboard with four charts: monthly sales trend, sales by region, sales by category, and the top 10 products.

In [7]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def generate_visual_report(df, path="Sales_Visual_Report.html"):
    monthly = df.groupby("Month")["Sales"].sum().reset_index()
    region = df.groupby("Region")["Sales"].sum().reset_index()
    cat = df.groupby("Category")["Sales"].sum().reset_index()
    top = df.groupby("Product Name")["Sales"].sum().nlargest(10).sort_values().reset_index()

    fig = make_subplots(rows=2, cols=2,
        subplot_titles=("Monthly Sales", "Sales by Region", "Sales by Category", "Top 10 Products"),
        specs=[[{}, {}], [{"type": "pie"}, {}]])
    fig.add_trace(go.Scatter(x=monthly["Month"], y=monthly["Sales"], mode="lines"), 1, 1)
    fig.add_trace(go.Bar(x=region["Region"], y=region["Sales"]), 1, 2)
    fig.add_trace(go.Pie(labels=cat["Category"], values=cat["Sales"]), 2, 1)
    fig.add_trace(go.Bar(x=top["Sales"], y=top["Product Name"], orientation="h"), 2, 2)
    fig.update_layout(height=800, showlegend=False, title_text="Automated Sales Report")
    fig.write_html(path)
    return fig

generate_visual_report(clean).show()

## One-click pipeline
The function `run_pipeline()` ties everything together. Given a CSV file, it:
1. Loads the data
2. Cleans it
3. Saves `cleaned_sales.csv`
4. Generates the Excel report and the visual report

To use it on a new file, run `run_pipeline("your_file.csv")`. The same steps run in seconds, instead of being repeated by hand every time.

In [8]:
def run_pipeline(input_csv):
    data = pd.read_csv(input_csv)
    cleaned, log = clean_data(data)
    cleaned.to_csv("cleaned_sales.csv", index=False)
    generate_excel_report(cleaned)
    generate_visual_report(cleaned)
    print("Cleaning log:", log)
    print("Files created: cleaned_sales.csv, Sales_Report.xlsx, Sales_Visual_Report.html")

run_pipeline("messy_sales.csv")

Cleaning log: {'rows_before': 10100, 'duplicates_removed': 144, 'bad_dates_dropped': 0, 'missing_sales_filled': 496, 'rows_after': 9956}
Files created: cleaned_sales.csv, Sales_Report.xlsx, Sales_Visual_Report.html


/tmp/ipykernel_1004/3035551854.py:10: FutureWarning:

Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.

/tmp/ipykernel_1004/3035551854.py:11: FutureWarning:

Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.

/tmp/ipykernel_1004/3035551854.py:12: FutureWarning:

Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.

/tmp/ipykernel_1004/3035551854.py:13: FutureWarning:

Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.



## Conclusion
- The pipeline removed [number] duplicates, fixed inconsistent text and date formats, and filled all missing values.
- Reports are generated automatically, which saves time and keeps the results consistent every time the data is updated.
- The cleaning rules can be reused for similar sales files by calling a single function.

**Limitations:** filling missing values with the median or most common value is an assumption and may not match the true values. For real business data, the rules should be reviewed with the data owner. The column names are also assumed to match this dataset.

**Tools used:** Python, Pandas, NumPy, Plotly, openpyxl.